In [ ]:
from opty import Problem, create_objective_function, parse_free
import sympy as sp
import numpy as np
import scipy as sc
import time as tm
import pickle
import sympy.physics.mechanics as me
import sys
sys.path.insert(0, "..")
from importlib import reload
import matplotlib.pyplot as plt
import equations as eq
reload (eq);
import trajectory_lib as tr
reload (tr);

participant = 'par2'
motion_list  =  ['All_motions']
GH_seq = 'YZY'

OS_struct = sc.io.loadmat('../Motions/'+participant+'/OS_model_prediction.mat')
act_w = 1

# build equations of motion, details are in muscle parameter calibration notebook
q,u,faux,fr,frstar,kinematical = eq.create_eoms_quat_w_RF(OS_struct,weight = 0,derive = 'numeric',gen_matlab_functions = 0)
clav_pos = 0.4

# we will iterate over 7 different simulations
# The first simulation is healthy case with GH stability term = 2
# Simulation 2-6 are with GH stability term = 2,4,6,8,10 and with the supra and infra damaged (activation set to 0)
# Simulation 7 is with non-caibrated muscle parameters
# Simulations are save in order in res_SHR_0 to res_SHR_6
wGHs = [2,2,4,6,8,10,2]
RC_lims = [1,0.01,0.01,0.01,0.01,0.01,1]

# Define inclination (around z-axis) and version (around y-axis) angles of the glenoid.
tilt_y = 13
tilt_z = -6.5
w_traj = 200
simulation = 'All_motions'
w_diff_vel = 1e-2
w_diff_exc = 1e-3
w_diff_faux = 1e-2
thor_hum_only = True

# List of muscles' activations corresponding to supra and infraspinatus muscle to be set to 0 during the RC-limited condition   
act_dmg = ['act_54(t)', 'act_55(t)', 'act_56(t)', 'act_57(t)', 'act_58(t)', 'act_59(t)', 'act_67(t)', 'act_68(t)', 'act_69(t)', 'act_70(t)']

for isim in range(len(wGHs)):

    wGH = wGHs[isim]
    RC_lim = RC_lims[isim]

    calibrated_params = sc.io.loadmat('../Motions/'+participant+'/calibrated_params.mat')
    if isim == 6:
        TE,activations,TE_conoid, fmax_init, fmax_range, lceopt_init, lceopt_range, mus_groups, GH_mus_forces = eq.polynomials_quat(OS_struct,q,u,calibrated_params = None, derive = 'numeric',RC_lim = RC_lim)
    else:    
        TE,activations,TE_conoid, fmax_init, fmax_range, lceopt_init, lceopt_range, mus_groups, GH_mus_forces = eq.polynomials_quat(OS_struct,q,u,calibrated_params = calibrated_params, derive = 'numeric',RC_lim = RC_lim)
    num_params = 0
    include_activation_dynamics = True
    optimize = 0

    eoms_implicit = sp.Matrix(kinematical).col_join(fr+frstar+sp.Matrix([TE+sp.Matrix(TE_conoid)]).col_join(GH_mus_forces))

    if include_activation_dynamics:
        excitations = []
        act_ode = []
        for i in range(len(activations)):
            excitations.append(me.dynamicsymbols('exc'+str(activations[i])[3:-3]))
            current_mus_ind = int(str(activations[i])[4:-3])
            current_mus = OS_struct['model']['muscles'].item()[0,(current_mus_ind-1)]
            t_act = current_mus['tact'][0,0].item()
            t_deact = current_mus['tdeact'][0,0].item()
            act_ode.append(activations[i].diff() - eq.act_dynamics(activations[i],excitations[i],t_act,t_deact))
        sp_act_ode = sp.Matrix(act_ode)
        eoms_implicit = eoms_implicit.col_join(sp_act_ode)

    interval_value = 0.04
    file = '../Motions/' + participant + '/' + simulation + '/' + simulation
    traj_original, omega, num_nodes, time = tr.exp_trajectory_quat(file,interval_value)
    q0_t0 = traj_original[:,0][:4]
    traj = tr.exp_trajectory_quat_myobj(traj_original,clav_pos)
    emg, indexes_emg = tr.exp_emg('../Motions/'+participant+'/'+simulation+'/EMG_'+participant+'_'+simulation+'.mat', num_nodes = num_nodes)

    # We exclude clavicle and scapula from the trajectory tracking objective
    index_clav_scap = 0
    # Humerus tracking is off during the pauses to let humerus find stable neutral position. 
    index_hum = indexes_emg
    

    if include_activation_dynamics:
        state_symbols = tuple(q+u+faux+activations)
        specified_symbols = tuple(excitations)
    else:
        state_symbols = tuple(q+u+faux)
        specified_symbols = tuple(activations)

    num_states = len(state_symbols) 
    num_q = len(q)
    num_u = len(u)
    num_faux = len(faux)
    num_inputs = len(specified_symbols)
    t = me.dynamicsymbols._t
    
    objective_traj,objective_traj_jac, objective_SC_t0, objective_SC_t0_jac = eq.objective_traj_quat(num_q,interval_value,clav_pos,True)
    objective_act,objective_act_jac = eq.objective_min_activation(activations,interval_value)
    objective_exc,objective_exc_jac = eq.objective_min_activation(activations,interval_value)
    objective_maxstab, objective_maxstab_jac = eq.objective_max_GH_stab(tilt_y=tilt_y,tilt_z=tilt_z,interval_value = interval_value)
    obj_min_diff,obj_min_diff_jac = eq.objective_state_diff(num_nodes,interval_value)

    def obj(free):
        min_traj = w_traj * np.sum(objective_traj(np.split(free[:num_q*num_nodes],num_q),traj,index_clav_scap,index_hum))
        min_SC_t0 = w_traj * np.sum(objective_SC_t0(free[0::num_nodes][:4],q0_t0))
        min_SC_tf = w_traj * np.sum(objective_SC_t0(free[num_nodes-1::num_nodes][:4],q0_t0))

        min_vel_dif = w_diff_vel * np.sum((obj_min_diff(np.transpose(np.split(free[num_q*num_nodes:(num_q + num_u)*num_nodes],num_u)))))
        min_faux_dif = w_diff_faux * np.sum((obj_min_diff(np.transpose(np.split(free[(num_q+num_u)*num_nodes:(num_q + num_u+num_faux)*num_nodes],num_faux)))))

        min_act = act_w * np.sum(objective_act(np.split(free[(num_q + num_u + num_faux)*num_nodes:(num_q + num_u + num_faux + num_inputs)*num_nodes],num_inputs)))

        min_instab = wGH * np.sum(objective_maxstab(np.split(free[(num_q + num_u)*num_nodes:(num_q + num_u + num_faux)*num_nodes],3)))

        obj = (min_traj + min_vel_dif + min_act + min_instab + min_faux_dif + min_SC_t0 + min_SC_tf) #   
        if include_activation_dynamics:
            min_exc_dif = w_diff_exc * np.sum((obj_min_diff(np.transpose(np.split(free[(num_states)*num_nodes:(num_states + num_inputs)*num_nodes],num_inputs)))))
            obj += (min_exc_dif)

        return obj.item()

    def obj_grad(free):
        grad = np.zeros_like(free)
        grad[:num_q*num_nodes] += w_traj * np.concatenate(objective_traj_jac(np.split(free[:num_q*num_nodes],num_q),traj,index_clav_scap,index_hum))
        grad[0::num_nodes][:4] += w_traj * np.sum(objective_SC_t0_jac(free[0::num_nodes][:4],q0_t0))
        grad[num_nodes-1::num_nodes][:4] += w_traj * np.sum(objective_SC_t0_jac(free[num_nodes-1::num_nodes][:4],q0_t0))

        grad[(num_q + num_u + num_faux)*num_nodes:(num_q + num_u + num_faux + num_inputs)*num_nodes] += act_w * np.concatenate(objective_act_jac(np.split(free[(num_q + num_u + num_faux)*num_nodes:(num_q + num_u + num_faux + num_inputs)*num_nodes],num_inputs)))

        grad[num_q*num_nodes:(num_q + num_u)*num_nodes] += w_diff_vel * np.concatenate(np.transpose((obj_min_diff_jac(np.transpose(np.split(free[num_q*num_nodes:(num_q + num_u)*num_nodes],num_u)))[0,:,:])))
        grad[(num_q+num_u)*num_nodes:(num_q + num_u+num_faux)*num_nodes] += w_diff_faux * np.concatenate(np.transpose((obj_min_diff_jac(np.transpose(np.split(free[(num_q+num_u)*num_nodes:(num_q + num_u+num_faux)*num_nodes],num_faux)))[0,:,:])))
        grad[(num_q + num_u)*num_nodes:(num_q + num_u + num_faux)*num_nodes] += wGH * np.concatenate(objective_maxstab_jac(np.split(free[(num_q + num_u)*num_nodes:(num_q + num_u + num_faux)*num_nodes],3)))

        if include_activation_dynamics:
            grad[(num_states)*num_nodes:(num_states + num_inputs)*num_nodes] += w_diff_exc * np.concatenate(np.transpose((obj_min_diff_jac(np.transpose(np.split(free[(num_states)*num_nodes:(num_states + num_inputs)*num_nodes],num_inputs)))[0,:,:])))

        return grad
    
    print('obj_check', obj(np.ones(num_states*num_nodes + num_inputs*num_nodes + num_params)*0.01))
    print('obj_grad_check', sum(obj_grad(np.ones(num_states*num_nodes + num_inputs*num_nodes + num_params)*0.01)))

    instance_constraints = []
        
    instance_constraints.append(state_symbols[13].func(time[-1]))
    instance_constraints.append(state_symbols[14].func(time[-1]))
    instance_constraints.append(state_symbols[15].func(time[-1]))
    instance_constraints.append(state_symbols[16].func(time[-1]))
    instance_constraints.append(state_symbols[17].func(time[-1]))
    instance_constraints.append(state_symbols[18].func(time[-1]))
    instance_constraints.append(state_symbols[19].func(time[-1]))
    instance_constraints.append(state_symbols[20].func(time[-1]))
    instance_constraints.append(state_symbols[21].func(time[-1]))
    instance_constraints.append(state_symbols[13].func(time[0]))
    instance_constraints.append(state_symbols[14].func(time[0]))
    instance_constraints.append(state_symbols[15].func(time[0]))
    instance_constraints.append(state_symbols[16].func(time[0]))
    instance_constraints.append(state_symbols[17].func(time[0]))
    instance_constraints.append(state_symbols[18].func(time[0]))
    instance_constraints.append(state_symbols[19].func(time[0]))
    instance_constraints.append(state_symbols[20].func(time[0]))
    instance_constraints.append(state_symbols[21].func(time[0]))
    instance_constraints.append(state_symbols[0].func(0)**2 + state_symbols[1].func(0)**2 + state_symbols[2].func(0)**2 + state_symbols[3].func(0)**2 - 1) # SC
    instance_constraints.append(state_symbols[4].func(0)**2 + state_symbols[5].func(0)**2 + state_symbols[6].func(0)**2 + state_symbols[7].func(0)**2 - 1) # AC
    instance_constraints.append(state_symbols[8].func(0)**2 + state_symbols[9].func(0)**2 + state_symbols[10].func(0)**2 + state_symbols[11].func(0)**2 - 1) # GH
        
    bounds1 = (0.0,1.0)
    bounds = (bounds1,)*len(activations)
    bndrs = dict(zip(activations,bounds))
    if include_activation_dynamics:
        bndrs_exc = dict(zip(excitations,(bounds1,)*len(excitations)))
        bndrs.update(bndrs_exc)


    for iact in activations:
        if str(iact) in act_dmg:
            if isim == 0 or isim == 6:
                bndrs.update({iact: (0.0, 1.0)})
            else:
                # Set the bounds of the damaged muscles' activations to 0 for simulations 2-6
                bndrs.update({iact: (0.0, 0.0)})
                
    for i in range(num_q):
        if i == 0 or i == 4 or i == 8:
            bndrs.update({q[i]: (min(traj_original[i,:])-0.2, 1.0)})
        else:
            bndrs.update({q[i]: (min(traj_original[i,:])-0.2, max(traj_original[i,:])+0.2)})

    bndrs.update({faux[0]: (-2,0)})

    start = tm.time()
    prob = Problem(obj, obj_grad, eoms_implicit, state_symbols,
                num_nodes, interval_value,
                known_parameter_map={},
                instance_constraints=instance_constraints,
                bounds=bndrs,
                integration_method='midpoint',
                parallel = False)


    time_to_create = tm.time() - start
    print(time_to_create)
    
    # Apart from these two options, we use the default settings of the solver.
    prob.add_option('limited_memory_max_history', 40)
    prob.add_option('max_iter',2000)
    
    # Initial guesses were set such the problm converges to feasible solution as it is very senstivite to the initial guess.
    if isim == 1:
        # For the RC-limited case with w_GH = 2, we use the trajectory and speeds from the experimental data, and set the faux force to -0.75 and the activations to the solution of the healthy case with w_GH = 2 as initial guess.
        initial_guess = np.ones(prob.num_free)*0.0
        initial_guess[:13*num_nodes] = traj_original.flatten()
        initial_guess[(num_q + num_u)*num_nodes:(num_q + num_u + 1)*num_nodes] = -0.75
        initial_guess[num_q * num_nodes : (num_q + num_u) * num_nodes] = omega.flatten()
    elif isim == 0:
        # For the healthy case, we use the solution of the calibration problem as initial guess.
        initial_guess = tr.initial_guess_from_solution('../Motions/'+participant+'/'+simulation+'/All_motions_params_calibration.mat',prob.num_free)[:prob.num_free]
    else:
        # For each next RC-limited case in the sensitivity analysis we use the previous solution as intial guess.
        initial_guess = tr.initial_guess_from_solution('../Motions/'+participant+'/'+simulation+'/res_SHR_' + str(isim-1) + '.mat',prob.num_free)[:prob.num_free]

    time_2_solve_start = tm.time()
    solution, info = prob.solve(initial_guess)
    time_2_solve = tm.time() - time_2_solve_start
    print(info['status_msg'])
    print(info['obj_val'])
    act_obj = np.sum(solution[num_states*num_nodes:(num_states + num_inputs)*num_nodes]**2)
    objective_value = prob.obj_value
    print('Objective activations: ', act_obj)

    # Save to matlab struct.
    file_name = '../Motions/'+participant+'/'+simulation+'/res_SHR_' + str(isim) + '.mat'
    tr.sol2struct(solution,activations,num_q,num_u,num_faux,num_inputs,num_nodes,time,objective_value,time_2_solve,file_name,include_activation_dynamics)
    # Save to .mot file.
    file_name_mot = '../Motions/'+participant+'/'+simulation+'/res_SHR_' + str(isim) + '.mot'
    tr.sol2mot_quat(solution, num_nodes, len(q), time, file_name_mot, GH_seq)